# FinanceAI - Fase 1: Simulación de Datos

El objetivo es generar los archivos `usuarios.json` y `transacciones.json` que servirán como semilla para todo el sistema (tanto para entrenar tus modelos como para que el Backend inicialice su base de datos).

### 📋 Reglas a recordar:
1. **Reproducibilidad:** Usar `SEED = 42` en todas las variables aleatorias.
2. **Fechas limpias:** Formato estricto `YYYY-MM-DD` sin horas ni minutos.
3. **Sin credenciales:** No generar emails ni contraseñas. Eso es responsabilidad del Backend.
4. **Categorías oficiales (9):** `Alimentacion`, `Transporte`, `Salud`, `Vivienda`, `Educacion`, `Ocio`, `Servicios`, `Deudas`, `Ahorros`.
5. **Tamaño de muestra:** Generaremos una muestra liviana (ej: 1,000 usuarios y unas 5,000 transacciones).

In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import json

# 1. Configuración de la semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

NUM_USUARIOS = 1000
NUM_TRANSACCIONES = 5000

## 1. Generación de Usuarios
Campos obligatorios: `id`, `nombre`, `ingreso_mensual`, `fecha_creacion`, `activo`, `nivel_endeudamiento`, `frecuencia_ahorro`, y `perfil_financiero`.

In [4]:
def generar_usuarios(n_usuarios):
    usuarios = []
    nombres_base = ['Juan', 'Maria', 'Pedro', 'Lucia', 'Carlos', 'Ana', 'Diego', 'Laura', 'Mateo', 'Sofia']
    frecuencias = ['Alta', 'Media', 'Baja', 'Nula']
    
    for i in range(n_usuarios):
        # Generar fecha aleatoria en el último año
        dias_restar = random.randint(0, 365)
        fecha_creacion = (datetime.now() - timedelta(days=dias_restar)).strftime('%Y-%m-%d')
        
        ingreso = round(random.uniform(500, 5000), 2)
        endeudamiento = round(random.uniform(0, 100), 2)
        frecuencia = random.choice(frecuencias)
        
        # Lógica heurística básica para simular el target (perfil_financiero)
        if endeudamiento > 40 or frecuencia == 'Nula':
            perfil = 'En riesgo'
        elif endeudamiento <= 40 and frecuencia in ['Alta', 'Media']:
            perfil = 'Saludable'
        else:
            perfil = 'En observacion'
            
        usuarios.append({
            'id': i,
            'nombre': f"{random.choice(nombres_base)} {i}",
            'ingreso_mensual': ingreso,
            'fecha_creacion': fecha_creacion,
            'activo': True,
            'nivel_endeudamiento': endeudamiento,
            'frecuencia_ahorro': frecuencia,
            'perfil_financiero': perfil
        })
        
    return pd.DataFrame(usuarios)

df_usuarios = generar_usuarios(NUM_USUARIOS)
df_usuarios.head()

,id,nombre,ingreso_mensual,fecha_creacion,activo,nivel_endeudamiento,frecuencia_ahorro,perfil_financiero
0,0,Lucia 0,1000.99,2025-09-03,True,74.16,Media,En riesgo
1,1,Sofia 1,3814.12,2026-05-17,True,67.67,Alta,En riesgo
2,2,Mateo 2,643.02,2025-12-23,True,9.37,Media,Saludable
3,3,Lucia 3,619.41,2025-09-22,True,19.88,Nula,En riesgo
4,4,Pedro 4,3151.70,2025-12-10,True,80.94,Alta,En riesgo


## 2. Generación de Transacciones
Simularemos el historial asociándolo a los IDs generados previamente. Los campos son: `fecha`, `user_id`, `nombre`, `descripcion`, `monto`, `tipo` y `categoria`.

In [5]:
def generar_transacciones(n_transacciones, df_usuarios):
    transacciones = []
    
    categorias_egreso = ['Alimentacion', 'Transporte', 'Salud', 'Vivienda', 'Educacion', 'Ocio', 'Servicios', 'Deudas', 'Ahorros']
    
    # Diccionario de DESCRIPCIONES por categoría para entrenar luego el modelo NLP
    textos_por_categoria = {
        'Alimentacion': ['Supermercado Coto', 'Carniceria', 'Verduleria', 'Carrefour', 'Panaderia'],
        'Transporte': ['Carga SUBE', 'Uber', 'Cabify', 'Estacion de servicio YPF', 'Peaje'],
        'Salud': ['Farmacity', 'Obra social', 'Consulta medica', 'Estudios clinicos'],
        'Vivienda': ['Alquiler', 'Expensas', 'Ferreteria', 'Muebles'],
        'Educacion': ['Cuota colegio', 'Libreria', 'Curso online Udemy', 'Universidad'],
        'Ocio': ['Netflix', 'Cine', 'Spotify', 'Restaurante', 'Bar', 'Juego Steam'],
        'Servicios': ['Edesur', 'Metrogas', 'Internet Personal', 'Factura Movistar'],
        'Deudas': ['Pago Tarjeta de Credito', 'Cuota prestamo', 'Intereses bancarios'],
        'Ahorros': ['Compra Dolar MEP', 'Fondo Comun Inversion', 'Plazo fijo']
    }
    
    for _ in range(n_transacciones):
        # Seleccionar un usuario al azar
        usuario = df_usuarios.sample(1).iloc[0]
        
        # Generar fecha aleatoria
        dias_restar = random.randint(0, 365)
        fecha_trx = (datetime.now() - timedelta(days=dias_restar)).strftime('%Y-%m-%d')
        
        # 90% egresos, 10% ingresos
        if random.random() > 0.10:
            tipo = 'egreso'
            categoria = random.choice(categorias_egreso)
            descripcion = random.choice(textos_por_categoria[categoria])
            monto = round(random.uniform(5, 500), 2)
        else:
            tipo = 'ingreso'
            categoria = 'Salario'
            descripcion = 'Acreditacion de Haberes'
            # Un ingreso suele ser mayor
            monto = round(random.uniform(1000, 5000), 2)
            
        transacciones.append({
            'fecha': fecha_trx,
            'user_id': int(usuario['id']),
            'nombre': usuario['nombre'],
            'descripcion': descripcion,
            'monto': monto,
            'tipo': tipo,
            'categoria': categoria
        })
        
    df_trx = pd.DataFrame(transacciones)
    # Ordenar por fecha cronológica ascendente
    df_trx = df_trx.sort_values(by='fecha').reset_index(drop=True)
    return df_trx

df_transacciones = generar_transacciones(NUM_TRANSACCIONES, df_usuarios)
df_transacciones.head()

,fecha,user_id,nombre,descripcion,monto,tipo,categoria
0,2025-07-27,816,Diego 816,Carga SUBE,74.26,egreso,Transporte
1,2025-07-27,750,Laura 750,Farmacity,498.95,egreso,Salud
2,2025-07-27,315,Ana 315,Estudios clinicos,382.67,egreso,Salud
3,2025-07-27,436,Juan 436,Bar,326.48,egreso,Ocio
4,2025-07-27,336,Lucia 336,Netflix,261.51,egreso,Ocio


## 3. Exportación de Datos
Finalmente exportamos los datos a formato CSV (para que los leas fácil en pandas más adelante) y a JSON (para entregar al equipo de Backend y que armen el Database Seeder).

In [ ]:
# Exportar a CSV
#df_usuarios.to_csv('usuarios.csv', index=False)
#f_transacciones.to_csv('transacciones.csv', index=False)

# Exportar a JSON (orient='records' crea el formato de array ideal para APIs/Backend)
#df_usuarios.to_json('usuarios.json', orient='records', indent=4)
#df_transacciones.to_json('transacciones.json', orient='records', indent=4)

print("¡Exportación exitosa! Los archivos están listos para compartir con Backend.")